In [9]:
from notebooks.summersoc.globals import PATH_METRICS_DEMO_EXPLORE, PATH_MODEL_LIST, SPLIT_DATA_INTO_X_PARTS, \
    ITERATE_THROUGH_X_PARTS
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib

plt.style.use('default')

from agent.components import RASK
from agent.components.GaussianProcess import GASK
from agent.components.commons import ServiceType, theoretical_param_bounds, ServiceVar, FIG_SIZE_DEFAULT
from agent.components.commons import SloSet

services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SloSet.DEFAULT, SloSet.HIGH_PERF, SloSet.LOW_COST, SloSet.HIGH_QUALITY]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
df_explore = pd.read_csv(PATH_METRICS_DEMO_EXPLORE)
df_explore_preprocessed = RASK.preprocess_data(df_explore)

gp_list = joblib.load(PATH_MODEL_LIST)['gp_list']
rm_list = joblib.load(PATH_MODEL_LIST)['rm_list']

INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str


## **Plan**: Intent-based Inference

Find optimal parameter configs for each SLO x Service combination after seeing certain data shares

**TODO**: This here could actually be the place where we introduce out multi-dimensional methodology

![Monitoring Infrastructure](img/RASK_Methodology.jpg)

### Infer parameter assignments

#### Variant 1: Using deterministic gradient descent

In [11]:
# TODO: Currently, this shows how the prediction accuracy improves across the training data.
#  But that's not what we want, we want it to use increasingly more data from the EXPLOIT data.
#  So the easiest thing will be to concatenate them and then just do the same as before.

from agent.components.Optimizer import run_optimizer_multi

solution_history = []

for i in range(ITERATE_THROUGH_X_PARTS):
    for q in slos:
        for s in services:
            data_ratio = (i + 1) / ITERATE_THROUGH_X_PARTS
            rm = rm_list[i]

            # solutions = run_optimizer_multi(s, q.value, gp, theoretical_param_bounds[s], runs=25)
            # run_optimizer_multi(s, q.value, rm, theoretical_param_bounds[s], runs=25)
            # solve_global(service_contexts, MAX_CORES, self.rask, self.last_assignments)

            # Run optimizer to find the best configuration
            solutions = run_optimizer_multi(s, q.value, rm, theoretical_param_bounds[s], runs=1)
            fitness, config = max(solutions, key=lambda x: x[0])

            # Predict performance (mu, sigma) for the chosen configuration
            x_state = {ServiceVar.COST: config[0], ServiceVar.QUALITY: config[1]}
            x_state = x_state | ({ServiceVar.MODEL: config[2]} if s == ServiceType.CV else {})
            pred = rm.predict(s, "max_tp", x_state)

            # Store everything needed for the next block
            # We include empirical_var_bounds here as it changes per iteration
            solution_history.append({
                'data_rate': data_ratio,
                'rep': None,
                'service_type': s.value,
                'slo_set': q.name,
                'p_fitness': fitness,
                'dist': pred,
                'cores': x_state[ServiceVar.COST],
                'data_quality': x_state[ServiceVar.QUALITY],
                'model_size': x_state[ServiceVar.MODEL] if s == ServiceType.CV else -1,
            })

            print(f"Optimal fitness for {s.value} and {q.name} with {data_ratio * 100}% training data: {fitness}")


Optimal fitness for elastic-workbench-qr-detector and DEFAULT with 10.0% training data: 0.7506364331033462
Optimal fitness for elastic-workbench-cv-analyzer and DEFAULT with 10.0% training data: 0.9755555555555557
Optimal fitness for elastic-workbench-pc-visualizer and DEFAULT with 10.0% training data: 0.67
Optimal fitness for elastic-workbench-qr-detector and HIGH_PERF with 10.0% training data: 0.9500000000000001
Optimal fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 10.0% training data: 0.9962962962962963
Optimal fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 10.0% training data: 0.9870370370370369
Optimal fitness for elastic-workbench-qr-detector and LOW_COST with 10.0% training data: 0.1
Optimal fitness for elastic-workbench-cv-analyzer and LOW_COST with 10.0% training data: 0.9333333333333296
Optimal fitness for elastic-workbench-pc-visualizer and LOW_COST with 10.0% training data: 0.766666666666666
Optimal fitness for elastic-workbench-qr-detector an

#### Variant 2: Using stochastic gradient descent

In [8]:

from agent.components.Optimizer import run_optimizer_multi

solution_history = []

for i in range(ITERATE_THROUGH_X_PARTS):
    for q in slos:
        for s in services:
            data_ratio = (i + 1) / ITERATE_THROUGH_X_PARTS
            gp = gp_list[i][s]

            # Run optimizer to find the best configuration
            solutions = run_optimizer_multi(s, q.value, gp, theoretical_param_bounds[s], runs=25)
            fitness, config = max(solutions, key=lambda x: x[0])

            # Predict performance (mu, sigma) for the chosen configuration
            x_state = {ServiceVar.COST: config[0], ServiceVar.QUALITY: config[1]}
            x_state = x_state | ({ServiceVar.MODEL: config[2]} if s == ServiceType.CV else {})
            mu, sigma = gp.predict(s, "max_tp", x_state)

            # Store everything needed for the next block
            # We include empirical_var_bounds here as it changes per iteration
            solution_history.append({
                'data_rate': data_ratio,
                'rep': None,
                'service_type': s.value,
                'slo_set': q.name,
                'p_fitness': fitness,
                'dist': (mu, sigma),
                'cores': x_state[ServiceVar.COST],
                'data_quality': x_state[ServiceVar.QUALITY],
                'model_size': x_state[ServiceVar.MODEL] if s == ServiceType.CV else -1,
            })

            print(f"Optimal fitness for {s.value} and {q.name} with {data_ratio * 100}% training data: {fitness}")

Optimal fitness for elastic-workbench-qr-detector and DEFAULT with 10.0% training data: 0.662502935486422
Optimal fitness for elastic-workbench-cv-analyzer and DEFAULT with 10.0% training data: 0.6338995068100465
Optimal fitness for elastic-workbench-pc-visualizer and DEFAULT with 10.0% training data: 0.9144441768274085
Optimal fitness for elastic-workbench-qr-detector and HIGH_PERF with 10.0% training data: 0.9486571592446512
Optimal fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 10.0% training data: 0.9418428366287145
Optimal fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 10.0% training data: 0.9870363358814629


KeyboardInterrupt: 

#### Export to candidate solution script, with each config repeated x times

In [4]:

repeatable_data = []
runs_per_config = 50

# Iterate through the list in steps of 3 (the size of your service triple)
for i in range(0, len(solution_history), 3):
    # Extract the current triple of rows
    triple = solution_history[i: i + 3]

    # Repeat this specific triple for the number of runs
    for run_idx in range(runs_per_config):
        for row in triple:
            new_row = row.copy()
            new_row['rep'] = run_idx + 1
            repeatable_data.append(new_row)

df_candidates = pd.DataFrame(repeatable_data)
df_candidates.to_csv(
    f'../statics/candidates/candidate_solutions_{ITERATE_THROUGH_X_PARTS}_{SPLIT_DATA_INTO_X_PARTS}_{runs_per_config}.csv',
    index=False)


OSError: Cannot save file into a non-existent directory: '../statics/candidates'